In [1]:
import numpy as np
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import EqualWeighted, MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

print(X_train.head())

                AAPL       AMD       BAC       BBY       CVX        GE  \
Date                                                                     
1990-01-03  0.007576 -0.030303  0.008045  0.118056 -0.016229 -0.001876   
1990-01-04  0.003759 -0.015500 -0.021355 -0.012422 -0.012831 -0.005639   
1990-01-05  0.003745 -0.031996 -0.021821  0.000000 -0.014855 -0.009452   
1990-01-08  0.003731  0.000000  0.005633 -0.075472  0.009424  0.005725   
1990-01-09 -0.007435  0.016527  0.000000  0.000000 -0.007469 -0.020803   

                  HD       JNJ       JPM        KO       LLY       MRK  \
Date                                                                     
1990-01-03  0.003581  0.004072  0.033589 -0.014318  0.000000  0.015896   
1990-01-04  0.006244  0.002028  0.003991 -0.004993 -0.005557 -0.015647   
1990-01-05 -0.013298 -0.010408  0.003975 -0.008212 -0.010874 -0.020641   
1990-01-08 -0.009883  0.016944  0.000000  0.021159  0.000000  0.012839   
1990-01-09 -0.026316 -0.031026 -0.031

In [2]:
#model
model = MeanRisk(
    risk_measure=RiskMeasure.CVAR,
    objective_function=ObjectiveFunction.MINIMIZE_RISK,
    portfolio_params=dict(name="Min CVaR"),
)
model.fit(X_train)
model.weights_


array([2.16562597e-02, 1.41765615e-12, 3.64939426e-13, 1.47358244e-02,
       1.35927719e-01, 1.91067310e-12, 3.58410853e-12, 2.10319534e-01,
       4.89865658e-13, 8.14734697e-02, 1.92817428e-02, 4.13677567e-12,
       7.79977277e-12, 1.26155640e-01, 4.04653978e-12, 1.52708774e-01,
       1.20106197e-02, 6.41727678e-03, 1.01024052e-01, 1.18289088e-01])

In [3]:
benchmark = EqualWeighted(portfolio_params=dict(name="Equal Weighted"))
benchmark.fit(X_train)
benchmark.weights_

array([0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05,
       0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

In [4]:
#prediction
pred_model = model.predict(X_test)
pred_bench = benchmark.predict(X_test)

In [5]:
np.asarray(pred_model)

array([ 0.00354605,  0.00342423, -0.00105085, ...,  0.01069377,
        0.00542224, -0.01217676])

In [6]:
print(pred_model.cvar)
print(pred_bench.cvar)

0.02174155209025761
0.025061083134673378


In [7]:
#analysis 
population = Population([pred_model, pred_bench])

In [8]:
population.plot_composition()

In [9]:
fig = population.plot_cumulative_returns()
show(fig)

In [10]:
population.summary()

,Min CVaR,Equal Weighted
Mean,0.052%,0.069%
Annualized Mean,13.07%,17.30%
Variance,0.0090%,0.012%
Annualized Variance,2.26%,2.94%
Semi-Variance,0.0046%,0.0060%
Annualized Semi-Variance,1.16%,1.52%
Standard Deviation,0.95%,1.08%
Annualized Standard Deviation,15.02%,17.15%
Semi-Deviation,0.68%,0.78%
Annualized Semi-Deviation,10.76%,12.32%


From the analysis on the test set, we see that the Minimum CVaR portfolio outperforms the equal-weighted benchmark for all deviation and shortfall risk measures.

